## Poisonous Mushroom Classification

This project predicts whether a mushroom is edible or poisonous from its physical characteristics (cap shape, color, gill features, stem measurements, habitat, season, and more). It is a binary classification problem where a wrong "edible" prediction on a poisonous mushroom is dangerous.

## Approach
1. Load the data and explore the target, feature types, and missing values
2. Build a pipeline that encodes the categorical features and trains a gradient boosting classifier
3. Evaluate MCC on a validation split
4. Generate predictions and create the submission file
5. Submit to Kaggle and record the score


In [1]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c playground-series-s4e8
!unzip -oq playground-series-s4e8.zip

Paste your Kaggle API token and press Enter: ··········
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 2.5 MB/s eta 0:00:00
100% 82.3M/82.3M [00:01<00:00, 84.0MB/s]



In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
train.shape, test.shape

((3116945, 22), (2077964, 21))

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3116945 entries, 0 to 3116944
Data columns (total 22 columns):
 #   Column                Dtype  
---  ------                -----  
 0   id                    int64  
 1   class                 object 
 2   cap-diameter          float64
 3   cap-shape             object 
 4   cap-surface           object 
 5   cap-color             object 
 6   does-bruise-or-bleed  object 
 7   gill-attachment       object 
 8   gill-spacing          object 
 9   gill-color            object 
 10  stem-height           float64
 11  stem-width            float64
 12  stem-root             object 
 13  stem-surface          object 
 14  stem-color            object 
 15  veil-type             object 
 16  veil-color            object 
 17  has-ring              object 
 18  ring-type             object 
 19  spore-print-color     object 
 20  habitat               object 
 21  season                object 
dtypes: float64(3), int64(1), object(18)
memory

In [5]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2077964 entries, 0 to 2077963
Data columns (total 21 columns):
 #   Column                Dtype  
---  ------                -----  
 0   id                    int64  
 1   cap-diameter          float64
 2   cap-shape             object 
 3   cap-surface           object 
 4   cap-color             object 
 5   does-bruise-or-bleed  object 
 6   gill-attachment       object 
 7   gill-spacing          object 
 8   gill-color            object 
 9   stem-height           float64
 10  stem-width            float64
 11  stem-root             object 
 12  stem-surface          object 
 13  stem-color            object 
 14  veil-type             object 
 15  veil-color            object 
 16  has-ring              object 
 17  ring-type             object 
 18  spore-print-color     object 
 19  habitat               object 
 20  season                object 
dtypes: float64(3), int64(1), object(17)
memory usage: 332.9+ MB


In [6]:
train.sample(1)

,id,class,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,...,stem-root,stem-surface,stem-color,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season
1442347,1442347,p,4.1,b,h,r,f,s,c,g,...,NaN,y,l,NaN,NaN,t,e,u,m,u


In [7]:
test.sample(1)

,id,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,...,stem-root,stem-surface,stem-color,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season
907904,4024849,5.09,c,k,y,f,NaN,c,y,5.7,...,NaN,NaN,w,NaN,NaN,f,f,NaN,d,a


In [8]:
train["class"].value_counts(normalize=True)

,proportion
class,
p,0.547137
e,0.452863


In [9]:
train.dtypes

,0
id,int64
class,object
cap-diameter,float64
cap-shape,object
cap-surface,object
cap-color,object
does-bruise-or-bleed,object
gill-attachment,object
gill-spacing,object
gill-color,object


In [10]:
train.isnull().sum()[lambda s: s > 0]

,0
cap-diameter,4
cap-shape,40
cap-surface,671023
cap-color,12
does-bruise-or-bleed,8
gill-attachment,523936
gill-spacing,1258435
gill-color,57
stem-root,2757023
stem-surface,1980861


In [11]:
test_ids = test["id"]
train = train.drop(columns=["id"]).sample(500000, random_state=42)
test = test.drop(columns=["id"])

In [12]:
x = train.drop(columns=["class"])
y = train["class"]

In [13]:
cat_cols = ["cap-shape", "cap-surface", "cap-color", "does-bruise-or-bleed", "gill-attachment", "gill-spacing", "gill-color", "stem-root", "stem-surface", "stem-color", "veil-type", "veil-color", "has-ring", "ring-type", "spore-print-color", "habitat", "season"]
num_cols = ["cap-diameter", "stem-height", "stem-width"]

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import train_test_split

prep = ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1, min_frequency=100), cat_cols)],
    remainder="passthrough",
)
model = Pipeline([
    ("prep", prep),
    ("clf", HistGradientBoostingClassifier(categorical_features=[True] * len(cat_cols) + [False] * len(num_cols), max_iter=200, random_state=42)),
])

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
model.fit(x_train, y_train)

print("Validation MCC:", round(matthews_corrcoef(y_val, model.predict(x_val)), 4))

Validation MCC: 0.9822


In [15]:
model.fit(x, y)

pred_test = model.predict(test)

pd.DataFrame({"id": test_ids, "class": pred_test}).to_csv("submission.csv", index=False)

In [16]:
!kaggle competitions submit -c playground-series-s4e8 -f submission.csv -m "HistGradientBoosting pipeline baseline"

100% 19.8M/19.8M [00:00<00:00, 26.1MB/s]
99 submissions remaining today.
Successfully submitted to Binary Prediction of Poisonous Mushrooms

In [17]:
import pickle, os
pickle.dump(model, open("mushroom_model.pkl", "wb"))
print("size MB:", round(os.path.getsize("mushroom_model.pkl") / 1e6, 2))

size MB: 1.0
